In [1]:
import numpy as np
import cv2
import torch
import os
import imageio
from deeplabv3 import DeepLabV3, SelfieConverter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mask_size = (257, 257)
model_path = 'models/deeplabv3.pth'

In [2]:
model = DeepLabV3().to(device)
model.load_state_dict(torch.load(model_path))
model = SelfieConverter(model)
model.eval()

SelfieConverter(
  (model): DeepLabV3(
    (avg_pool): AvgPool2d(kernel_size=33, stride=33, padding=0)
    (interp0): Upsample(size=[33, 33], mode='bilinear')
    (interp1): Upsample(size=[257, 257], mode='bilinear')
    (conv_block1): ConvBlock(
      (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16)
      (conv3): Conv2d(16, 8, kernel_size=(1, 1), stride=(1, 1))
    )
    (conv_block2): ConvBlock(
      (conv1): Conv2d(8, 48, kernel_size=(1, 1), stride=(1, 1))
      (conv2): Conv2d(48, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), groups=48)
      (conv3): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
    )
    (conv_block3): ConvBlock(
      (conv1): Conv2d(12, 72, kernel_size=(1, 1), stride=(1, 1))
      (conv2): Conv2d(72, 72, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=72)
      (conv3): Conv2d(72, 12, kernel_size=(1, 1), stride=(1, 1))
  

In [3]:
def load_gif_frames(gif_path, resolution):
    gif = imageio.mimread(gif_path)
    frames = []
    for frame in gif:
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        frame = cv2.resize(frame, resolution)
        frames.append(frame)
    return frames

In [4]:
def save_live_mask_from_camera():
    capture = cv2.VideoCapture(0)
    capture.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    resolution = (
        int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    )
    
    gif_frames = load_gif_frames('underwater.gif', resolution)
    gif_index = 0
    gif_length = len(gif_frames)


    while True:
        ret, frame = capture.read()
        if not ret:
            continue
        image = cv2.resize(frame, mask_size)
        x = image.transpose([2, 0, 1])[np.newaxis, ...]
        x = torch.from_numpy(x.astype(np.float32)).to(device) / 255.0
        with torch.no_grad():
            y = model(x)
        y = y[0, 0].cpu().numpy()
        mask = (y * 255).astype(np.uint8)
        mask = cv2.resize(mask, resolution)
        mask = cv2.medianBlur(mask, 5)
        
        cv2.imwrite('live_frame_cache/mask.png', mask)
        cv2.imwrite('live_frame_cache/frame.png', frame)

        foreground = cv2.imread('live_frame_cache/frame.png')
        alpha = cv2.imread('live_frame_cache/mask.png')
        
        background = gif_frames[gif_index]
        gif_index += 1
        
        if gif_index >= gif_length:
            gif_index = 0

        background = cv2.resize(background, resolution)

        foreground = foreground.astype(float)
        background = background.astype(float)

        alpha = alpha.astype(float)/255.0
        foreground = cv2.multiply(alpha, foreground)
        background = cv2.multiply(1.0 - alpha, background)
        composted_image = cv2.add(foreground, background)
        outimg = composted_image/255.0

        cv2.imshow('camera', outimg)
        # Exit key
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    capture.release()
    cv2.destroyAllWindows()

In [6]:
save_live_mask_from_camera()